# Ejercicio 2 — Variables casi duplicadas
**Universidad El Bosque — Programa de Matemáticas y Estadística — Introducción al Machine Learning — Semestre 2026-2**

Se analizan tres archivos, `experimento_2_a`, `_b` y `_c`, y en cada uno se comparan el modelo `x1 → y` y el modelo `(x1, x2) → y`, donde el segundo se escribe como $h(x)=\hat\beta_0+\hat\beta_1x_1+\hat\beta_2x_2$. La pregunta de fondo es qué ocurre con el riesgo y con los coeficientes cuando `x2` es casi una copia de `x1`.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import sys

sys.path.append("../src")

# Formato numérico uniforme para las tablas
pd.set_option("display.float_format", lambda v: f"{v:.6f}")

from lista4 import cargar_datos, resumen_regresion, correlacion

archivos = ["experimento_2_a", "experimento_2_b", "experimento_2_c"]

## (a), (b) y (c) Correlación, riesgos y coeficientes por archivo

Se recorren los tres archivos con exactamente las mismas funciones; solo cambia el nombre del archivo.

In [2]:
filas = {}
for nombre in archivos:
    d = cargar_datos(nombre)
    solo_x1 = resumen_regresion(d, ["x1"])
    ambas   = resumen_regresion(d, ["x1", "x2"])
    filas[nombre] = {
        "corr(x1, x2)": correlacion(d, "x1", "x2"),
        "R_x1":         solo_x1["riesgo"],
        "R_x1,x2":      ambas["riesgo"],
        "beta1":        ambas["coeficientes"]["x1"],
        "beta2":        ambas["coeficientes"]["x2"],
    }

tabla_2 = pd.DataFrame.from_dict(filas, orient="index")
tabla_2

,"corr(x1, x2)",R_x1,"R_x1,x2",beta1,beta2
experimento_2_a,0.999760,1.029350,1.025907,5.957231,-2.900650
experimento_2_b,0.999779,1.061539,1.059304,0.606315,2.325776
experimento_2_c,0.999821,1.141281,1.140041,4.903450,-1.911232


En los tres archivos la correlación entre `x1` y `x2` es prácticamente 1 (≈ 0.9998): `x2` es casi una copia de `x1`. Aun así, los coeficientes estimados del modelo con ambas variables son muy distintos de un archivo a otro (`beta1` pasa de unos 0.61 a unos 5.96, y `beta2` de unos −2.90 a unos 2.33), incluso con signos opuestos para `beta2`.

## (d) Mejora del riesgo $\Delta R=\hat R_{x_1}-\hat R_{x_1,x_2}$

In [3]:
tabla_2["delta_R"] = tabla_2["R_x1"] - tabla_2["R_x1,x2"]
tabla_2["delta_R / R_x1 (%)"] = 100 * tabla_2["delta_R"] / tabla_2["R_x1"]
tabla_2[["R_x1", "R_x1,x2", "delta_R", "delta_R / R_x1 (%)"]]

,R_x1,"R_x1,x2",delta_R,delta_R / R_x1 (%)
experimento_2_a,1.029350,1.025907,0.003443,0.334526
experimento_2_b,1.061539,1.059304,0.002235,0.210585
experimento_2_c,1.141281,1.140041,0.001240,0.108661


La mejora del riesgo es positiva (nunca puede ser negativa al agregar una variable a un modelo lineal ajustado por mínimos cuadrados con intercepto), pero es minúscula: del orden de milésimas, que representan entre ≈ 0.1 % y ≈ 0.3 % del riesgo del modelo de una sola variable.

## (e) Rango entre archivos de $\hat\beta_1$, $\hat\beta_2$ y $\Delta R$

In [4]:
def rango(z):
    """rango(z) = max(z) - min(z)"""
    return z.max() - z.min()

rangos = pd.Series({
    "rango(beta1)":   rango(tabla_2["beta1"]),
    "rango(beta2)":   rango(tabla_2["beta2"]),
    "rango(delta_R)": rango(tabla_2["delta_R"]),
}, name="Rango entre los tres archivos")
rangos.to_frame()

,Rango entre los tres archivos
rango(beta1),5.350916
rango(beta2),5.226426
rango(delta_R),0.002203


## (f) Interpretación

**¿La incorporación de `x2` produce un cambio importante en el riesgo empírico?** No. En los tres archivos $\Delta R$ es del orden de $10^{-3}$, y su rango entre archivos es de apenas ≈ 0.0022. Como `x2` es casi una copia de `x1`, no aporta información nueva que el modelo no tuviera ya, y por eso el riesgo prácticamente no baja.

**¿Los coeficientes cambian en la misma proporción entre los tres archivos?** No, y la diferencia es enorme. Mientras $\Delta R$ casi no se mueve (rango ≈ 0.002), los coeficientes oscilan en rangos de más de 5 unidades (≈ 5.35 para $\hat\beta_1$ y ≈ 5.23 para $\hat\beta_2$). Es decir, los coeficientes individuales son muy inestables mientras que el ajuste (el riesgo) es prácticamente idéntico.

Esto es el síntoma típico de la **multicolinealidad**: cuando `x1` y `x2` son casi la misma variable, los datos no permiten distinguir cuánto del efecto corresponde a cada una, y pequeñas variaciones en la muestra reparten el efecto de maneras muy distintas entre `beta1` y `beta2`. Lo que sí queda bien determinado es el efecto conjunto, como se verifica a continuación.

In [5]:
# Verificación complementaria: lo que sí es estable es la suma beta1 + beta2
tabla_2["beta1 + beta2"] = tabla_2["beta1"] + tabla_2["beta2"]
tabla_2[["beta1", "beta2", "beta1 + beta2"]]

,beta1,beta2,beta1 + beta2
experimento_2_a,5.957231,-2.900650,3.056581
experimento_2_b,0.606315,2.325776,2.932091
experimento_2_c,4.903450,-1.911232,2.992218


La suma $\hat\beta_1+\hat\beta_2$ es ≈ 3 en los tres archivos, aunque cada coeficiente por separado varíe tanto. Como $x_2\approx x_1$, el modelo se comporta como $\hat\beta_0+(\hat\beta_1+\hat\beta_2)x_1$: el predictor solo "ve" la suma, y por eso el riesgo es casi el mismo mientras el reparto individual es arbitrario.